# AIOps W2-D2: RCA Graph + Retrieval

Notebook nay import output D1, build service graph, retrieve similar incidents, classify RCA bang kNN-style va ghi `results/rca_output.json`.

In [1]:
from pathlib import Path
import json

from rca import (
    build_graph,
    build_rca_output,
    classify_knn,
    graph_temporal_top_k,
    load_alerts,
    load_json,
    retrieve_similar_incidents,
    validate_result,
    write_rca_output,
)

BASE_DIR = Path.cwd()
if not (BASE_DIR / "dataset").exists():
    candidate = BASE_DIR / "w2" / "d2"
    if (candidate / "dataset").exists():
        BASE_DIR = candidate

D1_SUMMARY = BASE_DIR.parent / "d1" / "results" / "cluster_summary.json"
ALERTS_PATH = BASE_DIR / "dataset" / "alerts_sample.jsonl"
SERVICES_PATH = BASE_DIR / "dataset" / "services.json"
INCIDENTS_PATH = BASE_DIR / "dataset" / "incidents_history.json"
OUTPUT_PATH = BASE_DIR / "results" / "rca_output.json"

print("Base:", BASE_DIR)
print("D1 summary:", D1_SUMMARY)

Base: C:\DA\AI-Ops\w2\d2
D1 summary: C:\DA\AI-Ops\w2\d1\results\cluster_summary.json


In [2]:
cluster_summary = load_json(D1_SUMMARY)
alerts = load_alerts(ALERTS_PATH)
graph_bundle = build_graph(SERVICES_PATH)
incidents = load_json(INCIDENTS_PATH)["incidents"]

print("clusters", len(cluster_summary["clusters"]))
print("alerts", len(alerts))
print("incidents", len(incidents))
print("graph nodes", len(graph_bundle["graph"]))

clusters 3
alerts 20
incidents 30
graph nodes 14


In [3]:
results = []
debug_rows = []

for cluster in cluster_summary["clusters"]:
    # Graph + temporal scorer: top-K candidate root causes.
    graph_top3 = graph_temporal_top_k(cluster, alerts, graph_bundle, top_k=3)

    # Retrieval: top-3 similar incidents by keyword/service similarity.
    similar = retrieve_similar_incidents(cluster, incidents, graph_top3, top_k=3)

    # kNN-style classifier: class + actions come from top-1 similar incident.
    classified = classify_knn(cluster, graph_top3, similar)

    result = {
        "cluster_id": cluster["cluster_id"],
        "graph_top3": graph_top3,
        **classified,
    }

    # Validate schema and fallback if retrieval is empty/invalid.
    result = validate_result(result, cluster)
    results.append(result)
    debug_rows.append({
        "cluster_id": cluster["cluster_id"],
        "graph_top1": graph_top3[0],
        "top_similar": [(item["id"], item["_similarity"]) for item in similar],
        "root_cause": result["root_cause"],
        "class": result["class"],
    })

rca_output = {"clusters_analyzed": len(results), "results": results}
debug_rows

[{'cluster_id': 'c-000-000',
  'graph_top1': ['payment-svc', 1.0],
  'top_similar': [('INC-2025-11-08', 1.0),
   ('INC-2025-09-05', 1.0),
   ('INC-2026-05-10', 1.0)],
  'root_cause': 'payment-svc',
  'class': 'connection_pool_exhaustion'},
 {'cluster_id': 'c-000-001',
  'graph_top1': ['recommender-svc', 0.92],
  'top_similar': [('INC-2025-08-02', 0.757),
   ('INC-2026-03-07', 0.709),
   ('INC-2025-10-28', 0.649)],
  'root_cause': 'recommender-svc',
  'class': 'memory_leak'},
 {'cluster_id': 'c-000-002',
  'graph_top1': ['search-svc', 0.92],
  'top_similar': [('INC-2026-05-25', 0.764),
   ('INC-2026-01-29', 0.68),
   ('INC-2025-09-21', 0.48)],
  'root_cause': 'search-svc',
  'class': 'cache_cold_start'}]

In [4]:
written = write_rca_output(D1_SUMMARY, ALERTS_PATH, SERVICES_PATH, INCIDENTS_PATH, OUTPUT_PATH)
print("Wrote", OUTPUT_PATH.relative_to(BASE_DIR))

required_fields = {"cluster_id", "graph_top3", "root_cause", "class", "confidence", "actions", "reasoning", "similar_incidents", "method"}
schema_ok = written["clusters_analyzed"] == len(written["results"]) and all(required_fields <= set(item) for item in written["results"])
print("schema_ok", schema_ok)

for deliverable in ["FINDINGS.md", "SUBMIT.md"]:
    print(f"{deliverable} exists={(BASE_DIR / deliverable).exists()}")

Wrote results\rca_output.json
schema_ok True
FINDINGS.md exists=True
SUBMIT.md exists=True


In [5]:
for item in written["results"]:
    similar = ", ".join(item["similar_incidents"])
    print(f"{item['cluster_id']} root={item['root_cause']} class={item['class']} confidence={item['confidence']} similar={similar}")

c-000-000 root=payment-svc class=connection_pool_exhaustion confidence=0.99 similar=INC-2025-11-08, INC-2025-09-05, INC-2026-05-10
c-000-001 root=recommender-svc class=memory_leak confidence=0.85 similar=INC-2025-08-02, INC-2026-03-07, INC-2025-10-28
c-000-002 root=search-svc class=cache_cold_start confidence=0.85 similar=INC-2026-05-25, INC-2026-01-29, INC-2025-09-21
